# MSMLS Library — TopKonfidence Evaluation

Consolidated notebook for the analyses in `MSMLS_lib/`. It reproduces the
figures that used to live in separate scripts under `scripts/` (kept there for reference),
now driven by a single **task ID** parameter and rendered inline.

**Folder layout after reorganisation:**
- `scripts/` — original standalone `.py` analysis scripts (unchanged, kept for reference)
- `data/ground_truth/` — MSMLS' validated standards table (`validated_with_GNPS_by_LibraryName.*`)
- `data/processing/` — cached task outputs (merged library-match TSV, scored annex summary, GT mapping table)
- `data/reprocessing/` — raw standards MGF/CSV used to build the ground-truth library
- `outputs/figures/` — static PNG/SVG figures (regenerated by this notebook)
- `outputs/legacy_html/` — interactive Plotly HTML from the original scripts (not regenerated here)
- `outputs/reports/` — text/CSV reports from `eval_ground_truth.py`

**Sections:**
1. Parameters & setup
2. Load + score library matches for `TASK_ID` (replaces the static `annex_summary_*.csv`)
3. Ground-truth ("Mise en forme") mapping and InChIKey resolution helpers
4. Ground-truth ANNOTATION accuracy report (`eval_ground_truth.py`)
5. Confusion analysis: app vs authentic standards (`confusion_matrix.py`)
6. App (multi-metric) vs top-cosine annotation (`cosine_vs_app.py`)
7. Match count × confidence label (`match_count_report.py`, `match_count_report_v2.py`)
8. Molecular mass × confidence label (`mol_mass_report.py`)
9. Scoring-parameter correlation matrix (`param_correlation_matrix.py`)
10. MS/MS mirror plots (`mirror_plot_*.py`)


## 1. Parameters & setup

In [ ]:
import sys, os, re, math, json
import warnings, logging

logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from difflib import SequenceMatcher

%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from bin.normalizer import _normalize_library_matches_dataframe, _clean_compound_name, _canonical_structure_key
from bin.scorer import _compute_scan_level_confidence

BASE   = os.getcwd()                      # MSMLS_lib
DATA   = os.path.join(BASE, "data")
PROC   = os.path.join(DATA, "processing")
GT_DIR = os.path.join(DATA, "ground_truth")
OUT_FIG = os.path.join(BASE, "outputs", "new_figures")
OUT_REP = os.path.join(BASE, "outputs", "reports")
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_REP, exist_ok=True)

print("Repo root:", REPO_ROOT)


### Parameters

`TASK_ID` is the single GNPS2 task whose library-match results drive every figure,
including the ground-truth accuracy report (§4) — it reuses `annex`/`merged` from §2
rather than fetching separately.

Set `USE_CACHED_DATA = False` to force a live re-fetch + re-score from GNPS2 instead of the
cached files under `data/processing/`.

In [ ]:
# ── notebook parameters ───────────────────────────────────────────────
TASK_ID          = "fe83e1b8c6a0494f85eaf6933f6d4bdf"   # single task id — drives every section
WORKFLOW_CHOICE  = "fbmn"
USE_CACHED_DATA  = False   # reuse data/processing/ cache instead of hitting GNPS2 live

GROUND_TRUTH_CSV   = os.path.join(GT_DIR, "validated_with_GNPS_by_LibraryName.csv")
GT_MAPPING_TSV     = os.path.join(PROC, "Mise en forme table final v3.tsv")


## 2. Load & score library matches for `TASK_ID`

Reproduces what `annex_summary_<hash>.csv` and `<task_id>-merged_results_with_gnps.tsv`
contain, either from the on-disk cache (fast, offline) or freshly from GNPS2 via the same
loader chain used by the Flask app (`taskresult` TSV → `workflow_fbmn` fallback), followed
by `_normalize_library_matches_dataframe` + `_compute_scan_level_confidence`.

In [ ]:
def fetch_raw_merged(task_id):
    """Raw (unnormalized) library-match rows for task_id, mirroring merged_results_with_gnps.tsv."""
    raw_df = None
    try:
        from gnpsdata import taskresult
        raw_df = taskresult.get_gnps2_task_resultfile_dataframe(
            task_id, "nf_output/library/merged_results_with_gnps.tsv"
        )
        if raw_df is None or len(raw_df) == 0:
            raw_df = None
        else:
            print(f"  Loaded {len(raw_df)} rows via taskresult TSV")
    except Exception as exc:
        print(f"  taskresult TSV failed: {exc}")

    if raw_df is None:
        try:
            from gnpsdata import workflow_fbmn
            raw_df = workflow_fbmn.get_library_match_dataframe(task_id, gnps2=True)
            if raw_df is None or len(raw_df) == 0:
                raw_df = None
            else:
                print(f"  Loaded {len(raw_df)} rows via workflow_fbmn")
        except Exception as exc:
            print(f"  workflow_fbmn failed: {exc}")

    if raw_df is None:
        raise RuntimeError(f"Could not load library matches for task {task_id}")
    return raw_df


def load_annex_and_merged(task_id, workflow_choice=WORKFLOW_CHOICE, use_cache=USE_CACHED_DATA):
    """Returns (annex, merged) dataframes for task_id — scored summary + raw merged rows.

    annex  ~ one row per scan, confidence score + component breakdown (annex_summary_*.csv)
    merged ~ per-hit library match rows incl. InChIKey-Planar / Compound_Name (…-merged_results_with_gnps.tsv).
             `merged` is always guaranteed to carry a real InChIKey-Planar column: some raw
             GNPS2 result files (e.g. plain taskresult/workflow_fbmn exports) omit it and only
             carry Smiles/INCHI, so it's derived via the same _canonical_structure_key logic
             the scorer itself uses, rather than left missing for downstream InChIKey lookups.
    """
    cache_hash    = task_id[:12]
    cached_annex  = os.path.join(PROC, f"annex_summary_{cache_hash}.csv")
    cached_merged = os.path.join(PROC, f"{task_id}-merged_results_with_gnps.tsv")

    if use_cache and os.path.exists(cached_annex) and os.path.exists(cached_merged):
        print(f"Using cached data/processing files for task {task_id}")
        annex  = pd.read_csv(cached_annex)
        merged = pd.read_csv(cached_merged, sep="\t")
        if "InChIKey-Planar" not in merged.columns:
            merged["InChIKey-Planar"] = merged.apply(_canonical_structure_key, axis=1)
        merged["#Scan#"] = merged["#Scan#"].astype(str)
        return annex, merged

    print(f"Fetching + scoring task {task_id} live from GNPS2 ...")
    raw_df = fetch_raw_merged(task_id)
    norm_df = _normalize_library_matches_dataframe(raw_df)
    if "InChIKey-Planar" not in norm_df.columns:
        norm_df["InChIKey-Planar"] = norm_df.apply(_canonical_structure_key, axis=1)
    annex = _compute_scan_level_confidence(norm_df, task_id=task_id, workflow_choice=workflow_choice)
    annex["scan"] = annex["scan"].astype(str)
    norm_df["#Scan#"] = norm_df["#Scan#"].astype(str)
    return annex, norm_df


annex, merged = load_annex_and_merged(TASK_ID)
print(f"\nannex:  {len(annex)} scored scans")
print(f"merged: {len(merged)} raw library-match rows")
annex.head()

## 3. Ground-truth mapping & InChIKey resolution helpers

`gt` is MSMLS' "Mise en forme" mapping of the 185 authentic-standard features (Mzmine
feature ID ↔ GNPS compound name ↔ SMILES). The resolver cascade below (shared across
sections 5–8) looks up the `InChIKey-Planar` for a compound name within a scan's candidate
pool in `merged`, handling energy-suffix stripping, case, and Massbank pipe-separated names.

In [ ]:
gt = pd.read_csv(GT_MAPPING_TSV, sep="\t")
gt["id_Mzmine"] = gt["id_Mzmine"].astype(str)
print(f"Ground-truth standards: {len(gt)} features")
gt.head()


In [ ]:
def _strip_energy_suffix(name):
    return re.sub(r"\s*[\||-]\s*\d+\.?\d*\s*(?:eV|ev)\s*$", "", str(name), flags=re.I).strip()


def _majority_ik(m):
    """Deterministic pick among a name-matched candidate set: the majority (mode)
    InChIKey-Planar, ties broken by first occurrence. Guards against duplicate-named
    library entries (e.g. one mislabeled outlier among several concordant hits) where
    an arbitrary .iloc[0] pick would be unstable across fetches."""
    return m["InChIKey-Planar"].value_counts().index[0]


def resolve_inchikey(scan_id, name, merged_df, scan_col="#Scan#"):
    """Resolve InChIKey-Planar for `name` within the candidate pool of `scan_id`.

    Cascade: exact/case-insensitive match -> energy-suffix-stripped match / prefix match
    -> pipe-fragment substring match (Massbank-style names) -> global lookup by name
    -> single-InChIKey pool fallback (all candidates in the scan agree on structure).
    Each step picks the majority InChIKey among its matches (see _majority_ik) rather
    than an arbitrary first row, since a single duplicate-named entry can carry a
    different (often mislabeled) structure than the rest of the group.
    """
    if pd.isna(name):
        return None
    sub = merged_df[merged_df[scan_col] == scan_id]
    sub_ik = sub[sub["InChIKey-Planar"].notna() & (sub["InChIKey-Planar"] != "UNKNOWN_STRUCTURE")]

    for cand in (str(name), str(name).upper()):
        m = sub_ik[sub_ik["Compound_Name"].str.upper() == cand.upper()]
        if len(m):
            return _majority_ik(m)

    base = _strip_energy_suffix(name)
    if base != name:
        m = sub_ik[sub_ik["Compound_Name"].str.upper() == base.upper()]
        if len(m):
            return _majority_ik(m)
        m = sub_ik[sub_ik["Compound_Name"].str.upper().str.startswith(base.upper())]
        if len(m):
            return _majority_ik(m)

    m = sub_ik[sub_ik["Compound_Name"].str.contains(str(name), case=False, na=False, regex=False)]
    if len(m):
        return _majority_ik(m)

    gm = merged_df.dropna(subset=["InChIKey-Planar"])
    gm = gm[gm["InChIKey-Planar"] != "UNKNOWN_STRUCTURE"]
    m = gm[gm["Compound_Name"].str.upper() == str(name).upper()]
    if len(m):
        return _majority_ik(m)

    pool = sub_ik["InChIKey-Planar"].unique()
    if len(pool) == 1:
        return pool[0]
    return None


# ── shared plotting palette (mirrors plusrise_scores/mw_analysis.py) ──
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#dcdbd6", "#ffffff"
CONF_ORDER = ["Consistent evidence", "Inconclusive", "Inconsistent evidence"]
CONF_COLOR = {
    "Consistent evidence":  "#1f77b4",
    "Inconclusive":          "#ff7f0e",
    "Inconsistent evidence": "#d62728",
}

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
})

rng = np.random.default_rng(42)


def violin_strip(ax, groups, labels, colors, ylabel="", title="", strip_alpha=0.3, strip_s=5, log_scale=False):
    """Violin + jittered strip overlay, shared by sections 7–8."""
    data_clean = [np.array(g, dtype=float) for g in groups]
    data_clean = [d[~np.isnan(d)] for d in data_clean]
    positions = list(range(len(labels)))

    parts = ax.violinplot(
        [d if len(d) > 1 else np.array([d[0], d[0]]) for d in data_clean if len(d)],
        positions=positions[:len(data_clean)], widths=0.55, showmedians=True, showextrema=True,
    )
    for pc, color in zip(parts["bodies"], colors):
        pc.set_facecolor(color); pc.set_alpha(0.35); pc.set_edgecolor("none")
    for part in ["cmedians", "cmins", "cmaxes", "cbars"]:
        if part in parts:
            parts[part].set_color("#333"); parts[part].set_linewidth(1.2)

    for i, (d, color) in enumerate(zip(data_clean, colors)):
        jitter = rng.uniform(-0.13, 0.13, len(d))
        ax.scatter(i + jitter, d, alpha=strip_alpha, s=strip_s, color=color, zorder=4, linewidths=0)
        if len(d):
            med = np.median(d)
            ax.text(i + 0.30, med, f"{med:.1f}", va="center", ha="left", fontsize=7.5, color="#222")

    if log_scale:
        ax.set_yscale("symlog", linthresh=1)
    ax.set_xticks(positions)
    short = [l.replace("evidence", "ev.").replace("Inconsistent", "Inconsist.") for l in labels]
    ax.set_xticklabels(short, fontsize=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.35, zorder=0)
    ax.set_axisbelow(True)


def panel_letter(ax, letter, dx=-0.09, dy=1.06):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=16, fontweight="bold",
            va="bottom", ha="left", color=INK)


def save_and_show(fig, name):
    svg = os.path.join(OUT_FIG, f"{name}.svg")
    png = os.path.join(OUT_FIG, f"{name}.png")
    fig.savefig(svg, bbox_inches="tight", dpi=180)
    fig.savefig(png, bbox_inches="tight", dpi=180)
    print(f"Saved -> {svg}")
    plt.show()


## 4. Ground-truth annotation accuracy report

Compares TopKonfidence's `top_compound` (and the count-based baseline) against MSMLS'
185 validated standards, using name normalisation + a synonym table + fuzzy matching to
classify each scan as `correct` / `correct_stereo` / `close` / `wrong` / unresolved.
Reuses `annex`/`merged` from §2 (same `TASK_ID`) rather than fetching separately.

In [ ]:
_STEREO_PREFIX = re.compile(
    r"^(?:l-|d-|dl-|r-|s-|\(r\)-|\(s\)-|\(\+\)-|\(-\)-|alpha-|beta-|"
    r"gamma-|delta-|cis-|trans-|endo-|exo-|meso-|n-|o-|p-|m-|"
    r"3-|2-|1-|4-|5-|6-)+",
    re.IGNORECASE,
)
_SALT_SUFFIX = re.compile(
    r"\s+(hydrochloride|hcl|sodium|potassium|calcium|acetate|sulfate|"
    r"phosphate|chloride|bromide|iodide|nitrate|citrate|fumarate|"
    r"maleate|succinate|tartrate|monohydrate|dihydrate|hydrate|salt)$",
    re.IGNORECASE,
)
_COLLISION_ENERGY = re.compile(r"\s*collision\s*energy\s*:\s*[\d.]+\s*$", re.IGNORECASE)
_EV_SUFFIX = re.compile(r"[\s\-]+\d+(?:\.\d+)?\s*ev(?:\s+unknown)?$", re.IGNORECASE)
_CANDIDATE_ANNOTATION = re.compile(r"^candidate\s+\S+.*?\(delta\s+mass:[^)]+\)$", re.IGNORECASE)
_KNOWN_ISOMERS_NOTE = re.compile(r"\s*\(known structural isomers:[^)]*\)", re.IGNORECASE)
_CAS_NUMBER = re.compile(r"^\d{2,7}-\d{2}-\d$")

_SYNONYMS = {
    "asparticacid": "aspartate", "glutamicacid": "glutamate", "succinicacid": "succinate",
    "fumaricacid": "fumarate", "maleicacid": "maleate", "malicacid": "malate",
    "citricacid": "citrate", "lacticacid": "lactate", "pyruvicacid": "pyruvate",
    "oxalicacid": "oxalate", "aceticacid": "acetate", "nicotinicacid": "nicotinate",
    "isonicotinicacid": "nicotinate", "pyridine3carboxylicacid": "nicotinate",
    "quinicacid": "quinate", "gluconicacid": "gluconate", "sorbicacid": "sorbate",
    "sebacanicacid": "sebacate", "sebacicacid": "sebacate", "glycolicacid": "glycolate",
    "glycericacid": "glycerate", "hydrocortisoneacetate": "cortisol21acetate",
    "cortisol21aceticacid": "cortisol21acetate", "hydrocortisone21acetate": "cortisol21acetate",
    "nadplus": "nadp", "nicotinamideadeninedinucleotidephosphate": "nadp",
    "triphosphopyridine": "nadp", "betanicotinamideadeninedinucleotidephosphate": "nadp",
    "nacetyl5hydroxytryptamine": "nacetylserotonin", "n1acetyl5hydroxytryptamine": "nacetylserotonin",
    "carbocysteine": "scarboxymethylcysteine", "lcarboxymethylcysteine": "scarboxymethylcysteine",
    "scarboxymethyllcysteine": "scarboxymethylcysteine", "26diaminoheptanedioate": "diaminopimelate",
    "26diaminoheptanedioicacid": "diaminopimelate", "26diaminopimeicacid": "diaminopimelate",
    "guanosine35cyclicmonophosphate": "cyclicgmp", "cgmp": "cyclicgmp",
    "nepsilonnepsilonnepsilontrimethyllysine": "nnntrimethyllysine", "trimethyllysine": "nnntrimethyllysine",
    "5aminoimidazole4carboxamide1betadribofuranosyl5monophosphate": "aicar",
    "5aminoimidazole4carboxamide1betaribofuranosyl5monophosphate": "aicar",
    "aicariboside5monophosphate": "aicar", "zmp": "aicar",
    "guanosinediphosphatemannose": "gdpmannose", "guanosine5diphosphomanose": "gdpmannose",
    "guanosine5diphosphomannose": "gdpmannose", "guanosine5diphosphodmannose": "gdpmannose",
}


def _apply_synonyms(norm):
    return _SYNONYMS.get(norm, norm)


def _normalise(name):
    if not isinstance(name, str):
        return ""
    name = name.strip()
    if _CANDIDATE_ANNOTATION.match(name):
        return "__candidate__"
    if _CAS_NUMBER.match(name.strip()):
        return "__cas__"
    name = _KNOWN_ISOMERS_NOTE.sub("", name)
    name = _clean_compound_name(name)
    name = _COLLISION_ENERGY.sub("", name)
    name = _EV_SUFFIX.sub("", name)
    name = _SALT_SUFFIX.sub("", name)
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]", "", name)
    return _apply_synonyms(name)


def _strip_stereo(name):
    while True:
        new = _STEREO_PREFIX.sub("", name, count=1).strip()
        if new == name:
            break
        name = new
    return name


def _similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


def _verdict(top, truth):
    """correct / correct_stereo / close (fuzzy>=0.82) / candidate / cas_number / wrong / no_data"""
    if isinstance(top, str) and _CANDIDATE_ANNOTATION.match(top.strip()):
        return "candidate"
    if isinstance(top, str) and _CAS_NUMBER.match(top.strip()):
        return "cas_number"

    top_n, truth_n = _normalise(top), _normalise(truth)
    if not top_n or not truth_n or top_n in ("__candidate__", "__cas__"):
        return "no_data"
    if top_n == truth_n or top_n in truth_n or truth_n in top_n:
        return "correct"

    top_s = _apply_synonyms(_normalise(_strip_stereo(top)))
    truth_s = _apply_synonyms(_normalise(_strip_stereo(truth)))
    if top_s and truth_s and (top_s == truth_s or top_s in truth_s or truth_s in top_s):
        return "correct_stereo"

    sim = max(_similarity(top_n, truth_n), _similarity(top_s, truth_s))
    return "close" if sim >= 0.82 else "wrong"


In [ ]:
print("Loading ground truth ...")
gt_eval = pd.read_csv(GROUND_TRUTH_CSV)
gt_eval.columns = gt_eval.columns.str.strip()
gt_eval["#Scan#"] = gt_eval["#Scan#"].astype(str)
gt_eval = gt_eval[["#Scan#", "PRIMARY_NAME"]].rename(columns={"#Scan#": "scan", "PRIMARY_NAME": "ground_truth"})
print(f"  {len(gt_eval)} validated scans in ground truth")

gt_annex = annex.copy()
gt_annex["scan"] = gt_annex["scan"].astype(str)

eval_merged = gt_annex.merge(gt_eval, on="scan", how="inner")
scans_no_match = len(gt_eval) - len(eval_merged)
print(f"  {len(eval_merged)} scans overlap with ground truth ({scans_no_match} GT scans unmatched)")

eval_merged["verdict"] = eval_merged.apply(lambda r: _verdict(r["top_compound"], r["ground_truth"]), axis=1)
eval_merged["verdict_count_based"] = eval_merged.apply(
    lambda r: _verdict(r["count_based_top_compound"], r["ground_truth"]), axis=1
)

out_cols = [c for c in [
    "scan", "ground_truth", "top_compound", "count_based_top_compound",
    "verdict", "verdict_count_based", "confidence_score", "confidence_label",
    "supporting_matches", "total_matches", "tanimoto_score", "tanimoto_source",
    "max_cosine", "cosine_consistency", "shared_peaks", "mz_precision",
    "top_cosine_overridden", "top_cosine_compound",
] if c in eval_merged.columns]
eval_merged[out_cols].to_csv(os.path.join(OUT_REP, "eval_results.csv"), index=False)
print("Saved:", os.path.join(OUT_REP, "eval_results.csv"))


In [ ]:
total = len(eval_merged)
vc = eval_merged["verdict"].value_counts()
correct   = vc.get("correct", 0) + vc.get("correct_stereo", 0)
close     = vc.get("close", 0)
wrong     = vc.get("wrong", 0)
candidate = vc.get("candidate", 0)
cas       = vc.get("cas_number", 0)

pct = lambda n: f"{100*n/total:.1f}%" if total else "-"

print("=" * 70)
print("  TopKonfidence — Ground Truth Accuracy Report")
print(f"  Task : {TASK_ID}")
print(f"  Ground-truth compounds  : {len(gt_eval):>4}")
print(f"  Overlap (scored + GT)   : {total:>4}")
print(f"  GT scans with NO match  : {scans_no_match:>4}  ({100*scans_no_match/len(gt_eval):.1f}% missed entirely)")
print("=" * 70)
print()
print("ANNOTATION ACCURACY  (top_compound vs ground_truth)")
print(f"  Correct (incl. stereo variants) : {correct:>4}  ({pct(correct)})")
print(f"  Close (fuzzy sim >= 0.82)        : {close:>4}  ({pct(close)})")
print(f"  Wrong (different compound)       : {wrong:>4}  ({pct(wrong)})")
print(f"  Candidate/acyl annotation        : {candidate:>4}  ({pct(candidate)})  [unresolved name]")
print(f"  CAS number (no name)             : {cas:>4}  ({pct(cas)})  [unresolved name]")
print()

print("ACCURACY BY CONFIDENCE CATEGORY")
for cat in CONF_ORDER:
    sub = eval_merged[eval_merged["confidence_label"] == cat]
    if len(sub) == 0:
        continue
    n = len(sub)
    c  = sub["verdict"].isin(["correct", "correct_stereo"]).sum()
    w  = (sub["verdict"] == "wrong").sum()
    print(f"  {cat:<25} n={n:>3}  correct={c:>3} ({100*c/n:.0f}%)  wrong={w:>3} ({100*w/n:.0f}%)")


## 5. Confusion analysis: app vs authentic standards

Three outcomes per scan: `Agree`, `Disagree — GT in candidates` (recoverable), or
`Disagree — GT absent`, stratified by confidence label. Also breaks out the evidence-based
score vs top-cosine-only 2×2 win/loss comparison.

In [ ]:
def resolve_gt_inchikey(row, merged_df):
    """Resolve GT InChIKey via name-matching against the candidate pool, falling back
    to the ground-truth table's own SMILES when the name lookup fails (e.g. verbose
    GNPS_Compound_Name strings like "Spectral Match to X from NIST14" that don't
    fuzzy-match shorter candidate names in the pool)."""
    ik = resolve_inchikey(row["id_Mzmine"], row["GNPS_Compound_Name"], merged_df)
    if ik is not None:
        return ik
    smiles = row.get("SMILES")
    if pd.notna(smiles) and str(smiles).strip():
        key = _canonical_structure_key({"Smiles": smiles})
        if key and key != "UNKNOWN_STRUCTURE":
            return key
    return None


gt_c = gt.copy()
gt_c["gt_inchikey"] = [resolve_gt_inchikey(r, merged) for _, r in gt_c.iterrows()]
print(f"GT InChIKey resolved: {gt_c['gt_inchikey'].notna().sum()}/{len(gt_c)}")

scan_pool = (
    merged[merged["#Scan#"].isin(gt_c["id_Mzmine"])]
    .groupby("#Scan#")["InChIKey-Planar"].apply(lambda x: set(x.dropna()))
)

df5 = gt_c.merge(
    annex[["scan", "top_compound", "structure_key", "confidence_label", "confidence_score",
           "top_cosine_compound", "top_cosine_overridden"]],
    left_on="id_Mzmine", right_on="scan", how="inner",
)

def outcome(row):
    gt_ik, app_ik = row["gt_inchikey"], row["structure_key"]
    if pd.isna(gt_ik) or pd.isna(app_ik):
        return "Unresolved"
    if gt_ik == app_ik:
        return "Agree"
    pool = scan_pool.get(row["id_Mzmine"], set())
    return "Disagree – GT in candidates" if gt_ik in pool else "Disagree – GT absent"

df5["outcome"] = df5.apply(outcome, axis=1)
df5["confidence_label"] = pd.Categorical(df5["confidence_label"], categories=CONF_ORDER, ordered=True)

df5["cosine_ik"] = df5.apply(
    lambda r: r["structure_key"] if not r["top_cosine_overridden"]
              else resolve_inchikey(r["id_Mzmine"], r["top_cosine_compound"], merged),
    axis=1,
)
df5["app_correct"]    = df5["gt_inchikey"] == df5["structure_key"]
df5["cosine_correct"] = df5["cosine_ik"].notna() & (df5["gt_inchikey"] == df5["cosine_ik"])

n_gt = len(df5)
app_correct_n = int(df5["app_correct"].sum())
cos_correct_n = int(df5["cosine_correct"].sum())
print(f"\nEvidence-based score accuracy: {app_correct_n}/{n_gt} ({app_correct_n/n_gt*100:.1f}%)")
print(f"Top cosine only accuracy:      {cos_correct_n}/{n_gt} ({cos_correct_n/n_gt*100:.1f}%)")
print("\n=== OUTCOME SUMMARY ===")
print(df5["outcome"].value_counts())


In [ ]:
OUTCOME_ORDER = ["Agree", "Disagree – GT in candidates", "Disagree – GT absent"]
OUTCOME_COLORS = {
    "Agree": "#008300", "Disagree – GT in candidates": "#eda100",
    "Disagree – GT absent": "#e34948", "Unresolved": "#a8a7a2",
}
OUTCOME_SHORT = {
    "Agree": "Agree", "Disagree – GT in candidates": "Disagree, in pool",
    "Disagree – GT absent": "Disagree, absent", "Unresolved": "Unresolved",
}
present_outcomes = [o for o in OUTCOME_ORDER if o in df5["outcome"].values]
all_outcomes = present_outcomes + (["Unresolved"] if "Unresolved" in df5["outcome"].values else [])

ct_plot = (df5.groupby(["confidence_label", "outcome"], observed=True).size()
             .unstack(fill_value=0).reindex(index=CONF_ORDER, columns=OUTCOME_ORDER, fill_value=0))
ct_plot["Unresolved"] = (df5[df5["outcome"] == "Unresolved"]
                          .groupby("confidence_label", observed=True).size()
                          .reindex(CONF_ORDER, fill_value=0))
matrix = ct_plot[OUTCOME_ORDER]
matrix_pct = matrix.div(matrix.sum(axis=1), axis=0) * 100
disagree = df5[df5["outcome"].str.startswith("Disagree")].copy()
disagree_sorted = disagree.sort_values(["outcome", "confidence_score"], ascending=[True, False])


def draw_outcome_bar(ax_bar):
    y = np.arange(len(CONF_ORDER))
    totals = ct_plot.sum(axis=1).reindex(CONF_ORDER).values
    left = np.zeros(len(CONF_ORDER))
    for out_label in all_outcomes:
        counts = ct_plot.get(out_label, pd.Series(0, index=CONF_ORDER)).reindex(CONF_ORDER, fill_value=0).values
        pct_ = np.divide(counts, totals, out=np.zeros_like(counts, dtype=float), where=totals > 0) * 100
        ax_bar.barh(y, pct_, left=left, height=0.62, color=OUTCOME_COLORS[out_label],
                    label=OUTCOME_SHORT[out_label], edgecolor=SURFACE, linewidth=0.8, zorder=3)
        for yi, p, l, n in zip(y, pct_, left, counts):
            if n > 0 and p >= 6:
                ax_bar.text(l + p / 2, yi, str(int(n)), ha="center", va="center",
                            fontsize=8.5, fontweight="bold", color=SURFACE, zorder=4)
        left = left + pct_
    for yi, tot in zip(y, totals):
        ax_bar.text(104, yi, f"n={int(tot)}", ha="left", va="center", fontsize=7.5, color=INK_2, clip_on=False)
    ax_bar.set_xlim(0, 100); ax_bar.set_xticks([0, 25, 50, 75, 100])
    ax_bar.set_xlabel("Scans (%)", fontsize=8.5)
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels([c.replace(" evidence", "\nevidence") for c in CONF_ORDER], fontsize=8)
    ax_bar.invert_yaxis()
    ax_bar.set_title("Outcome vs authentic standards,\nby evidence tier", fontsize=10, fontweight="bold", loc="left", pad=22)
    ax_bar.legend(frameon=False, ncol=2, fontsize=7, loc="lower left", bbox_to_anchor=(-0.02, 1.0),
                  columnspacing=1.0, handlelength=1.3, handletextpad=0.5)
    ax_bar.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax_bar.set_axisbelow(True); ax_bar.spines["left"].set_visible(False); ax_bar.tick_params(axis="y", length=0)


def draw_accuracy_heatmap(ax_heat):
    im = ax_heat.imshow(matrix_pct.values, cmap="Greens", aspect="auto", vmin=0, vmax=100)
    for i in range(len(CONF_ORDER)):
        for j in range(len(OUTCOME_ORDER)):
            n_, p_ = int(matrix.values[i, j]), matrix_pct.values[i, j]
            ax_heat.text(j, i, f"{n_}\n({p_:.0f}%)", ha="center", va="center",
                         fontsize=8.5, color=SURFACE if p_ > 60 else INK)
    ax_heat.set_xticks(range(len(OUTCOME_ORDER)))
    ax_heat.set_xticklabels([OUTCOME_SHORT[o].replace(", ", "\n") for o in OUTCOME_ORDER], fontsize=8)
    ax_heat.set_yticks(range(len(CONF_ORDER)))
    ax_heat.set_yticklabels([c.replace(" evidence", "\nevidence") for c in CONF_ORDER], fontsize=8)
    ax_heat.tick_params(length=0)
    ax_heat.set_title("Per-tier outcome\nbreakdown", fontsize=10, fontweight="bold", loc="left", pad=8)
    cbar = plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)
    cbar.set_label("Row %", fontsize=8.5); cbar.ax.tick_params(labelsize=7.5)


def draw_disagreement_list(ax_list):
    ax_list.axis("off")
    ax_list.set_title(f"Disagreement cases\n(n = {len(disagree_sorted)})", fontsize=10, fontweight="bold", loc="left", pad=8)
    n_cases = len(disagree_sorted)
    row_h = 0.95 / max(n_cases, 1)
    for ri, (_, row) in enumerate(disagree_sorted.iterrows()):
        yi = 0.98 - ri * row_h
        is_pool = "candidates" in row["outcome"]
        marker_color = OUTCOME_COLORS["Disagree – GT in candidates" if is_pool else "Disagree – GT absent"]
        conf_color = CONF_COLOR.get(row["confidence_label"], INK_2)
        ax_list.scatter([0.015], [yi], s=20, color=marker_color, zorder=3, transform=ax_list.transAxes, clip_on=False)
        ax_list.text(0.06, yi, str(row["PRIMARY_NAME"]).title(), transform=ax_list.transAxes,
                     fontsize=7.8, fontweight="bold", color=INK, va="top")
        predicted = str(row["top_compound"]).title().replace("?", "'") if pd.notna(row["top_compound"]) else "—"
        ax_list.text(0.06, yi - row_h * 0.34, f"→ annotation explorer: {predicted}",
                     transform=ax_list.transAxes, fontsize=7.2, color=INK_2, va="top")
        ax_list.text(0.06, yi - row_h * 0.63,
                     f"{row['confidence_label']} ({row['confidence_score']:.0f})  ·  {'recoverable' if is_pool else 'not in pool'}",
                     transform=ax_list.transAxes, fontsize=6.8, color=conf_color, va="top")
    ax_list.set_xlim(0, 1); ax_list.set_ylim(0, 1)


def draw_cosine_comparison_2x2(ax):
    ax.axis("off")
    ax.set_title("Evidence-based score vs\ntop cosine only", fontsize=10, fontweight="bold", loc="left", pad=8)
    both_correct = int((df5["app_correct"] & df5["cosine_correct"]).sum())
    app_only     = int((df5["app_correct"] & ~df5["cosine_correct"]).sum())
    cos_only     = int((~df5["app_correct"] & df5["cosine_correct"]).sum())
    both_wrong   = int((~df5["app_correct"] & ~df5["cosine_correct"]).sum())
    mat = np.array([[both_correct, cos_only], [app_only, both_wrong]])
    cell_labels = [["Both correct", "Evidence ✗\nTop match ✓"], ["Evidence ✓\nTop match ✗", "Both wrong"]]
    cell_colors = [["#d9f2d9", "#fdecc8"], ["#dce8fb", "#f9d6d6"]]
    ax.set_xlim(0, 2); ax.set_ylim(0, 2.35)
    for ri in range(2):
        for ci in range(2):
            x, y0 = ci, 1 - ri
            ax.add_patch(plt.Rectangle((x + 0.04, y0 + 0.04), 0.92, 0.86,
                                        facecolor=cell_colors[ri][ci], edgecolor=GRID, linewidth=1))
            v = mat[ri, ci]
            ax.text(x + 0.5, y0 + 0.58, str(v), ha="center", va="center", fontsize=15, fontweight="bold", color=INK)
            ax.text(x + 0.5, y0 + 0.36, f"{v / n_gt * 100:.0f}%", ha="center", va="center", fontsize=8, color=INK_2)
            ax.text(x + 0.5, y0 + 0.16, cell_labels[ri][ci], ha="center", va="center", fontsize=6.3, color=INK_2, style="italic")
    ax.text(0.5, 2.18, "GNPSAnnex evidence\n(top 10 matches) correct", ha="center", va="center", fontsize=7.5, fontweight="bold", color=INK)
    ax.text(1.5, 2.18, "GNPSAnnex evidence\n(top 10 matches) wrong", ha="center", va="center", fontsize=7.5, fontweight="bold", color=INK)
    ax.text(-0.09, 1.44, "Standard workflow\n(top match) correct", ha="center", va="center", fontsize=7.5, fontweight="bold", rotation=90, color=INK)
    ax.text(-0.09, 0.44, "Standard workflow\n(top match) wrong", ha="center", va="center", fontsize=7.5, fontweight="bold", rotation=90, color=INK)


fig = plt.figure(figsize=(9.2, 7.2))
gs = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[1, 1], height_ratios=[1, 1],
                        wspace=0.45, hspace=0.55, top=0.90, bottom=0.07, left=0.09, right=0.97)
ax_a, ax_b, ax_c, ax_d = (fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
                          fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]))
draw_outcome_bar(ax_a); draw_accuracy_heatmap(ax_b); draw_cosine_comparison_2x2(ax_c); draw_disagreement_list(ax_d)
for ax, letter in zip((ax_a, ax_b, ax_c, ax_d), "abcd"):
    panel_letter(ax, letter, dx=-0.14, dy=1.18)

save_and_show(fig, "confusion_figure_v2")


## 6. App (multi-metric) vs top-cosine annotation

Focuses on the cases where `top_cosine_overridden=True` — where the multi-metric score
picked a different compound than raw top-cosine similarity would have — and checks which
one actually matches ground truth.

In [ ]:
df6 = gt.merge(
    annex[["scan", "top_compound", "structure_key", "top_cosine_compound",
           "top_cosine_overridden", "confidence_label", "confidence_score"]],
    left_on="id_Mzmine", right_on="scan", how="inner",
)
df6["gt_ik"] = gt_c["gt_inchikey"].values if len(gt_c) == len(df6) else [
    resolve_gt_inchikey(r, merged) for _, r in df6.iterrows()
]

df6["cosine_ik"] = df6.apply(
    lambda r: r["structure_key"] if not r["top_cosine_overridden"]
              else resolve_inchikey(r["id_Mzmine"], r["top_cosine_compound"], merged),
    axis=1,
)
df6["app_correct"]    = df6["gt_ik"] == df6["structure_key"]
df6["cosine_correct"] = df6["cosine_ik"].notna() & (df6["gt_ik"] == df6["cosine_ik"])
df6["cosine_has_ans"] = df6["cosine_ik"].notna()

def quadrant(row):
    a, c, h = row["app_correct"], row["cosine_correct"], row["cosine_has_ans"]
    if a and c: return "Both correct"
    if a and not h: return "App ✓  |  Cosine: no answer"
    if a and not c: return "App ✓  |  Cosine ✗"
    if not a and c: return "App ✗  |  Cosine ✓"
    return "Both wrong"

df6["quadrant"] = df6.apply(quadrant, axis=1)
print(df6["quadrant"].value_counts())

n6 = len(df6)
app_acc6, cosine_acc6 = df6["app_correct"].sum(), df6["cosine_correct"].sum()
print(f"\nApp accuracy:    {app_acc6}/{n6} = {app_acc6/n6*100:.1f}%")
print(f"Cosine accuracy: {cosine_acc6}/{n6} = {cosine_acc6/n6*100:.1f}%")

ov6 = df6[df6["top_cosine_overridden"]].copy()
print(f"\nOverridden cases: {len(ov6)}")


In [ ]:
QUAD_COLOR = {
    "Both correct": "#2ca02c", "App ✓  |  Cosine: no answer": "#1f77b4",
    "App ✗  |  Cosine ✓": "#ff7f0e", "Both wrong": "#d62728",
}
QUAD_BG = {
    "Both correct": "#f0fff4", "App ✓  |  Cosine: no answer": "#eff6ff",
    "App ✗  |  Cosine ✓": "#fff7ed", "Both wrong": "#fff0f0",
}

fig = plt.figure(figsize=(14, 9))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38, width_ratios=[1, 1.6])
ax_mat, ax_list = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

mat_vals = {
    (True, True):  df6[(df6["app_correct"]) & (df6["cosine_correct"])].shape[0],
    (True, False): df6[(df6["app_correct"]) & (~df6["cosine_correct"])].shape[0],
    (False, True): df6[(~df6["app_correct"]) & (df6["cosine_correct"])].shape[0],
    (False, False):df6[(~df6["app_correct"]) & (~df6["cosine_correct"])].shape[0],
}
mat6 = np.array([[mat_vals[(True, True)], mat_vals[(False, True)]],
                  [mat_vals[(True, False)], mat_vals[(False, False)]]])
cell_colors = [["#d4edda", "#fff3cd"], ["#d1ecf1", "#f8d7da"]]
ax_mat.set_xlim(0, 2); ax_mat.set_ylim(0, 2.9); ax_mat.axis("off")
ax_mat.set_title("A  |  App vs cosine accuracy (2×2)", fontsize=12, fontweight="bold", loc="left", pad=10)
labels = [["Both correct", "App ✗\nCosine ✓"], ["App ✓\nCosine ✗\n(or no answer)", "Both wrong"]]
for ri in range(2):
    for ci in range(2):
        x, y = ci, 1 - ri
        rect = mpatches.FancyBboxPatch((x + 0.05, y + 0.05), 0.9, 0.85, boxstyle="round,pad=0.03",
                                        facecolor=cell_colors[ri][ci], edgecolor="#ccc", linewidth=1.2)
        ax_mat.add_patch(rect)
        v = mat6[ri, ci]
        ax_mat.text(x + 0.50, y + 0.56, str(v), ha="center", va="center", fontsize=22, fontweight="bold")
        ax_mat.text(x + 0.50, y + 0.23, f"{v/n6*100:.1f}%", ha="center", va="center", fontsize=11, color="#555")
        ax_mat.text(x + 0.50, y + 0.10, labels[ri][ci], ha="center", va="center", fontsize=8, color="#666", style="italic")
ax_mat.text(0.50, 1.97, "App correct", ha="center", va="center", fontsize=10, fontweight="bold")
ax_mat.text(1.50, 1.97, "App wrong", ha="center", va="center", fontsize=10, fontweight="bold")
ax_mat.text(-0.08, 1.48, "Cosine\ncorrect", ha="center", va="center", fontsize=10, fontweight="bold", rotation=90)
ax_mat.text(-0.08, 0.48, "Cosine\nwrong", ha="center", va="center", fontsize=10, fontweight="bold", rotation=90)
ax_mat.axhline(1.0, xmin=0.04, xmax=0.96, color="#aaa", linewidth=1.2)
ax_mat.axvline(1.0, ymin=0.04, ymax=0.72, color="#aaa", linewidth=1.2)

ax_mat.text(0.5, 2.45, "Overall accuracy", ha="center", va="center", fontsize=10, fontweight="bold")
bar_y = 2.20
for label, acc, color, xoff in [("App", app_acc6/n6*100, "#1f77b4", 0.30), ("Cosine", cosine_acc6/n6*100, "#ff7f0e", 1.20)]:
    bw = 0.38 * acc / 100
    ax_mat.add_patch(mpatches.FancyBboxPatch((xoff, bar_y), bw, 0.18, boxstyle="round,pad=0.01", facecolor=color, edgecolor="none"))
    ax_mat.text(xoff - 0.02, bar_y + 0.09, label, ha="right", va="center", fontsize=9, color="#333")
    ax_mat.text(xoff + bw + 0.02, bar_y + 0.09, f"{acc:.1f}% ({int(acc*n6/100)}/{n6})", ha="left", va="center", fontsize=9, color="#333")

ax_list.axis("off")
ax_list.set_title(f"B  |  Cases where app and cosine DISAGREE\n({len(ov6)} cases have top_cosine_overridden=True)",
                  fontsize=11, fontweight="bold", loc="left")
ov6_sorted = ov6.sort_values("quadrant")
headers = ["GT compound", "App annotation", "Top cosine", "Outcome", "Conf. (score)"]
col_x = [0.00, 0.22, 0.45, 0.68, 0.82]
y0, row_h = 0.95, 0.083
for h, xp in zip(headers, col_x):
    ax_list.text(xp, y0, h, transform=ax_list.transAxes, fontsize=8, fontweight="bold", va="top", color="#222")
ax_list.plot([0, 1], [y0 - 0.018, y0 - 0.018], transform=ax_list.transAxes, color="#aaa", linewidth=0.8)
CONF_C = {"Consistent evidence": "#1f77b4", "Inconclusive": "#d97706", "Inconsistent evidence": "#dc2626"}
for ri, (_, row) in enumerate(ov6_sorted.iterrows()):
    yi = y0 - (ri + 1) * row_h - 0.02
    bg = QUAD_BG.get(row["quadrant"], "#fff")
    ax_list.add_patch(mpatches.FancyBboxPatch((0, yi - row_h * 0.18), 1, row_h * 0.88, boxstyle="round,pad=0.004",
                       transform=ax_list.transAxes, facecolor=bg, edgecolor="none", zorder=0))
    tc = str(row["top_cosine_compound"]) if pd.notna(row["top_cosine_compound"]) else "—"
    app = str(row["top_compound"]) if pd.notna(row["top_compound"]) else "—"
    outcome_short = ("✓✓ Both correct" if row["quadrant"] == "Both correct" else
                      "✓ App | – Cosine" if "no answer" in row["quadrant"] else
                      "✗ App | ✓ Cosine" if "Cosine ✓" in row["quadrant"] else "✗ Both wrong")
    out_color = ("#16a34a" if "Both correct" in row["quadrant"] else "#2563eb" if "no answer" in row["quadrant"]
                 else "#ea580c" if "Cosine ✓" in row["quadrant"] else "#dc2626")
    conf_c = CONF_C.get(row["confidence_label"], "#666")
    cells6 = [(str(row["PRIMARY_NAME"])[:22], "#111"), (app[:22], "#333"), (tc[:22], "#333"),
              (outcome_short, out_color), (f"{str(row['confidence_label'])[:12]} ({row['confidence_score']:.0f})", conf_c)]
    for (v, vc), xp in zip(cells6, col_x):
        ax_list.text(xp, yi, v, transform=ax_list.transAxes, fontsize=7.2, va="top", color=vc)
legend_patches = [mpatches.Patch(color=QUAD_BG[k], label=k) for k in ["Both correct", "App ✓  |  Cosine: no answer", "App ✗  |  Cosine ✓"]]
ax_list.legend(handles=legend_patches, loc="lower right", fontsize=8, frameon=True, title="Row color")

fig.suptitle("App (multi-metric) vs top-cosine annotation — MSMLS library", fontsize=13, fontweight="bold", y=1.01)
save_and_show(fig, "cosine_vs_app_v2")


## 7. Match count × confidence label

Does the number of retrieved library matches explain `Inconsistent evidence`? Categorises
those scans into single-hit / few-hit / capped-at-10, each split by whether structural
consensus was already perfect, and shows which score component was actually the bottleneck.

In [ ]:
annex7 = annex.copy()
annex7["support_fraction"] = annex7["supporting_matches"] / annex7["total_matches"]
MATCH_CAP = 10

incon = annex7[annex7["confidence_label"] == "Inconsistent evidence"].copy()
single = incon["total_matches"] == 1
at_cap = incon["total_matches"] == MATCH_CAP
few_fragmented = (~single) & (~at_cap) & (incon["support_fraction"] < 1.0)
few_consensus  = (~single) & (~at_cap) & (incon["support_fraction"] == 1.0)
cap_consensus  = at_cap & (incon["support_fraction"] == 1.0)
cap_fragmented = at_cap & (incon["support_fraction"] < 1.0)

CAT_LABELS = ["Single hit (n=1)", "2–9 hits, full consensus", "2–9 hits, fragmented",
              "10 hits (cap), full consensus", "10 hits (cap), fragmented"]
CAT_MASKS = [single, few_consensus, few_fragmented, cap_consensus, cap_fragmented]
CAT_COLORS = ["#e15759", "#f28e2b", "#edc948", "#76b7b2", "#4e79a7"]
cat_counts = [m.sum() for m in CAT_MASKS]

print("=== Inconsistent evidence cause breakdown ===")
for lbl, n in zip(CAT_LABELS, cat_counts):
    print(f"  {lbl}: {n}  ({n/len(incon)*100:.1f}%)" if len(incon) else f"  {lbl}: {n}")

SCORE_COMPONENTS = {
    "Structure\nagreement": "support_fraction", "Tanimoto": "tanimoto_score",
    "Cosine\nconsistency": "cosine_consistency", "Max cosine": "max_cosine",
    "Shared peaks": "shared_peaks_score", "M/Z precision": "mz_precision",
}
comp_matrix = np.full((len(CAT_LABELS), len(SCORE_COMPONENTS)), np.nan)
for ri, mask in enumerate(CAT_MASKS):
    sub = incon[mask]
    for ci, col in enumerate(SCORE_COMPONENTS.values()):
        if col in sub.columns:
            comp_matrix[ri, ci] = sub[col].median()


In [ ]:
fig = plt.figure(figsize=(17, 13))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.52, wspace=0.38, height_ratios=[1.05, 1.0])
ax_A, ax_B, ax_C = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2])
ax_D, ax_E = fig.add_subplot(gs[1, 0:2]), fig.add_subplot(gs[1, 2])

violin_strip(ax_A, [annex7[annex7["confidence_label"] == l]["total_matches"] for l in CONF_ORDER],
             CONF_ORDER, [CONF_COLOR[l] for l in CONF_ORDER],
             ylabel="Total retrieved matches", title="A  |  Match count by confidence label\n(capped at 10 per scan)")
ax_A.axhline(MATCH_CAP, color="#999", linestyle="--", linewidth=1, zorder=2)
ax_A.text(2.48, MATCH_CAP + 0.05, "cap = 10", va="bottom", ha="right", fontsize=7.5, color="#888")

violin_strip(ax_B, [annex7[annex7["confidence_label"] == l]["support_fraction"] for l in CONF_ORDER],
             CONF_ORDER, [CONF_COLOR[l] for l in CONF_ORDER],
             ylabel="Support fraction (supporting / total)", title="B  |  Structural consensus fraction\nby confidence label")
ax_B.set_ylim(-0.05, 1.12)
if len(incon):
    pct_perfect = (incon["support_fraction"] == 1.0).mean() * 100
    ax_B.text(2, 1.08, f"{pct_perfect:.0f}% have\nfraction=1.0", ha="center", va="bottom", fontsize=7.5,
              color=CONF_COLOR["Inconsistent evidence"], style="italic")

ax_C.set_title(f"C  |  What drives 'Inconsistent evidence'?\n(n={len(incon)} scans)", fontsize=10, fontweight="bold", loc="left")
ax_C.axis("off")
total_incon = max(len(incon), 1)
left = 0.0
bar_patches = []
for lbl, n, color in zip(CAT_LABELS, cat_counts, CAT_COLORS):
    w = n / total_incon
    ax_C.add_patch(mpatches.Rectangle((left, 0.15), w, 0.55, transform=ax_C.transAxes,
                                       facecolor=color, edgecolor="white", linewidth=1.5, zorder=3))
    if w > 0.08:
        ax_C.text(left + w / 2, 0.42, f"{n}\n({n/total_incon*100:.0f}%)", transform=ax_C.transAxes,
                  ha="center", va="center", fontsize=8, fontweight="bold", color="white", zorder=4)
    bar_patches.append(mpatches.Patch(color=color, label=f"{lbl}: {n} ({n/total_incon*100:.0f}%)"))
    left += w
ax_C.legend(handles=bar_patches, loc="lower left", bbox_to_anchor=(0, -0.05), fontsize=7.5, frameon=True, title="Category")

for lbl in CONF_ORDER:
    sub = annex7[annex7["confidence_label"] == lbl]
    jitter = rng.uniform(-0.3, 0.3, len(sub))
    ax_D.scatter(sub["total_matches"] + jitter, sub["confidence_score"], alpha=0.2, s=7,
                 color=CONF_COLOR[lbl], label=lbl, linewidths=0, zorder=3)
for n_match in range(1, MATCH_CAP + 1):
    sub = annex7[annex7["total_matches"] == n_match]["confidence_score"]
    if len(sub):
        ax_D.plot(n_match, sub.median(), "D", color="#222", markersize=5, zorder=6, alpha=0.85)
ax_D.plot([], [], "D", color="#222", markersize=5, label="Median per match count")
ax_D.axhline(80, color="#1f77b4", linestyle=":", linewidth=1, alpha=0.7)
ax_D.axhline(55, color="#d62728", linestyle=":", linewidth=1, alpha=0.7)
ax_D.set_xlabel("Total retrieved matches (jittered)", fontsize=9)
ax_D.set_ylabel("Confidence score", fontsize=9)
ax_D.set_title(f"D  |  Confidence score vs match count (all {len(annex7)} scans)\nDiamonds = median per match count",
               fontsize=10, fontweight="bold", loc="left")
ax_D.legend(fontsize=8, loc="lower right", frameon=True, markerscale=1.5)
ax_D.set_xticks(range(1, MATCH_CAP + 1))
ax_D.spines[["top", "right"]].set_visible(False)
ax_D.yaxis.grid(True, linestyle="--", alpha=0.35, zorder=0); ax_D.set_axisbelow(True)

comp_labels = list(SCORE_COMPONENTS.keys())
im = ax_E.imshow(comp_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
plt.colorbar(im, ax=ax_E, label="Median value [0–1]", fraction=0.046, pad=0.04)
for ri in range(len(CAT_LABELS)):
    for ci in range(len(comp_labels)):
        v = comp_matrix[ri, ci]
        txt = f"{v:.2f}" if not np.isnan(v) else "—"
        color = "white" if (not np.isnan(v)) and (v < 0.35 or v > 0.80) else "black"
        ax_E.text(ci, ri, txt, ha="center", va="center", fontsize=7.5, color=color)
ax_E.set_xticks(range(len(comp_labels))); ax_E.set_xticklabels(comp_labels, fontsize=7.5, rotation=30, ha="right")
short_cat = [l.replace("(cap)", "(cap)\n").replace("consensus", "cons.") for l in CAT_LABELS]
ax_E.set_yticks(range(len(CAT_LABELS))); ax_E.set_yticklabels(short_cat, fontsize=7.5)
ax_E.set_title("E  |  Median score components\nper Inconsistent category", fontsize=10, fontweight="bold", loc="left")

fig.suptitle(f"Retrieved match count × confidence label analysis\nTopKonfidence · MSMLS library · task {TASK_ID[:12]}",
             fontsize=13, fontweight="bold", y=1.00)
save_and_show(fig, "match_count_report_v2")


## 8. Molecular mass × confidence label

Annotated / ground-truth compound MW (via RDKit `ExactMolWt` on the InChIKey-Planar's
first known SMILES) and query peak count, stratified by confidence label and by
`top_cosine_overridden` status.

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False
    print("WARNING: rdkit not found; MW panels will be empty.")


def smiles_to_mw(smi):
    if not RDKIT_OK or pd.isna(smi) or not str(smi).strip():
        return float("nan")
    try:
        mol = Chem.MolFromSmiles(str(smi))
        return Descriptors.ExactMolWt(mol) if mol else float("nan")
    except Exception:
        return float("nan")


ik_to_smiles = (
    merged.dropna(subset=["InChIKey-Planar", "Smiles"]).groupby("InChIKey-Planar")["Smiles"].first().to_dict()
)

annex8 = annex.copy()
annex8["annotated_smiles"] = annex8["structure_key"].map(ik_to_smiles)
annex8["annotated_mw"] = annex8["annotated_smiles"].apply(smiles_to_mw)
print(f"Annotated MW computed: {annex8['annotated_mw'].notna().sum()}/{len(annex8)}")

gt8 = gt.copy()
gt8["gt_mw"] = gt8["SMILES"].apply(smiles_to_mw)
print(f"GT MW computed: {gt8['gt_mw'].notna().sum()}/{len(gt8)}")

gt_annex8 = annex8[annex8["scan"].isin(gt8["id_Mzmine"])].merge(
    gt8[["id_Mzmine", "PRIMARY_NAME", "SMILES", "gt_mw", "GNPS_Compound_Name"]],
    left_on="scan", right_on="id_Mzmine", how="left",
)

gt_annex8["gt_inchikey"] = [
    resolve_inchikey(r["scan"], r["GNPS_Compound_Name"], merged) for _, r in gt_annex8.iterrows()
]
gt_annex8["outcome"] = gt_annex8.apply(
    lambda r: "Agree" if (pd.notna(r["gt_inchikey"]) and pd.notna(r["structure_key"])
                          and r["gt_inchikey"] == r["structure_key"]) else "Disagree",
    axis=1,
)
disagree_gt8 = gt_annex8[gt_annex8["outcome"] == "Disagree"].copy()
print(f"\nGT disagreements: {len(disagree_gt8)}")

annex8["override_label"] = annex8["top_cosine_overridden"].map({True: "Overridden", False: "Not overridden"})
OVR_COLOR = {"Not overridden": "#2ca02c", "Overridden": "#9467bd"}


In [ ]:
def violin_strip_mw(ax, groups, labels, colors, ylabel="", title="", strip_alpha=0.35, strip_s=6):
    data_clean = [np.array(g, dtype=float) for g in groups]
    data_clean = [d[~np.isnan(d)] for d in data_clean]
    positions = list(range(len(labels)))
    if any(len(d) > 1 for d in data_clean):
        parts = ax.violinplot([d if len(d) > 1 else np.array([d[0], d[0]]) for d in data_clean],
                               positions=positions, widths=0.55, showmedians=True, showextrema=True)
        for pc, color in zip(parts["bodies"], colors):
            pc.set_facecolor(color); pc.set_alpha(0.35); pc.set_edgecolor("none")
        for part in ["cmedians", "cmins", "cmaxes", "cbars"]:
            if part in parts:
                parts[part].set_color("#333"); parts[part].set_linewidth(1.2)
    for i, (d, color) in enumerate(zip(data_clean, colors)):
        jitter = rng.uniform(-0.12, 0.12, len(d))
        ax.scatter(i + jitter, d, alpha=strip_alpha, s=strip_s, color=color, zorder=4, linewidths=0)
        if len(d):
            med = np.median(d)
            ax.text(i + 0.28, med, f"{med:.0f}", va="center", ha="left", fontsize=7.5, color="#222")
    ax.set_xticks(positions)
    short_labels = [l.replace("evidence", "ev.").replace("Inconsistent", "Inconsist.") for l in labels]
    ax.set_xticklabels(short_labels, fontsize=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.35, zorder=0); ax.set_axisbelow(True)


fig = plt.figure(figsize=(17, 14))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38, height_ratios=[1.0, 1.0, 1.1])
ax_A, ax_B, ax_C = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2])
ax_D, ax_E, ax_F = fig.add_subplot(gs[1, 0:2]), fig.add_subplot(gs[1, 2]), fig.add_subplot(gs[2, 0:3])

violin_strip_mw(ax_A, [annex8[annex8["confidence_label"] == l]["annotated_mw"].dropna() for l in CONF_ORDER],
                CONF_ORDER, [CONF_COLOR[l] for l in CONF_ORDER], ylabel="Exact MW (Da)",
                title=f"A  |  Annotated compound MW\nby confidence label  (n={len(annex8)})")
violin_strip_mw(ax_B, [gt_annex8[gt_annex8["confidence_label"] == l]["gt_mw"].dropna() for l in CONF_ORDER],
                CONF_ORDER, [CONF_COLOR[l] for l in CONF_ORDER], ylabel="Exact MW (Da)",
                title=f"B  |  Ground-truth compound MW\nby confidence label  (n={len(gt_annex8)})")
violin_strip_mw(ax_C, [annex8[annex8["confidence_label"] == l]["query_peak_count"].dropna() for l in CONF_ORDER],
                CONF_ORDER, [CONF_COLOR[l] for l in CONF_ORDER], ylabel="Filtered query peak count",
                title=f"C  |  Query spectrum peak count\nby confidence label  (n={len(annex8)})")

for lbl in CONF_ORDER:
    sub = annex8[annex8["confidence_label"] == lbl].dropna(subset=["annotated_mw", "query_peak_count"])
    ax_D.scatter(sub["annotated_mw"], sub["query_peak_count"], alpha=0.25, s=8, color=CONF_COLOR[lbl],
                 label=lbl, linewidths=0, zorder=3)
corr_data = annex8.dropna(subset=["annotated_mw", "query_peak_count"])
if len(corr_data) > 1:
    r = np.corrcoef(corr_data["annotated_mw"], corr_data["query_peak_count"])[0, 1]
    ax_D.text(0.97, 0.97, f"r = {r:.3f}", transform=ax_D.transAxes, ha="right", va="top", fontsize=9, color="#333",
              bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#ccc", alpha=0.8))
ax_D.set_xlabel("Annotated compound exact MW (Da)", fontsize=9)
ax_D.set_ylabel("Filtered query peak count", fontsize=9)
ax_D.set_title("D  |  MW vs query peak count (all scans)", fontsize=10, fontweight="bold", loc="left")
ax_D.legend(title="Confidence", fontsize=8, title_fontsize=8, loc="upper right", frameon=True, markerscale=2)
ax_D.spines[["top", "right"]].set_visible(False)
ax_D.yaxis.grid(True, linestyle="--", alpha=0.35, zorder=0); ax_D.set_axisbelow(True)

ovr_groups = [annex8[annex8["override_label"] == l]["annotated_mw"].dropna() for l in ["Not overridden", "Overridden"]]
n_not_ovr = (annex8["override_label"] == "Not overridden").sum()
n_ovr = (annex8["override_label"] == "Overridden").sum()
ovr_labels = [f"Not overridden\n(n={n_not_ovr})", f"Overridden\n(n={n_ovr})"]
violin_strip_mw(ax_E, ovr_groups, ovr_labels, [OVR_COLOR["Not overridden"], OVR_COLOR["Overridden"]],
                ylabel="Exact MW (Da)", title="E  |  MW: override vs not\n(top_cosine_overridden)")

ax_F.axis("off")
ax_F.set_title("F  |  GT disagreement cases — molecular context", fontsize=10, fontweight="bold", loc="left", pad=6)
headers = ["Scan", "GT compound", "App annotation", "GT MW (Da)", "App MW (Da)", "ΔMW", "Query peaks", "Shared peaks", "Confidence (score)"]
col_x = [0.00, 0.08, 0.28, 0.47, 0.56, 0.65, 0.73, 0.82, 0.90]
y0, row_h = 0.91, 0.15
for h, xp in zip(headers, col_x):
    ax_F.text(xp, y0, h, transform=ax_F.transAxes, fontsize=7.5, fontweight="bold", va="top", color="#111")
ax_F.plot([0, 1], [y0 - 0.03, y0 - 0.03], transform=ax_F.transAxes, color="#aaa", linewidth=0.8)
disagree_sorted8 = disagree_gt8.sort_values("confidence_score", ascending=False)
for ri, (_, row) in enumerate(disagree_sorted8.iterrows()):
    yi = y0 - (ri + 1) * row_h - 0.04
    ax_F.add_patch(mpatches.FancyBboxPatch((0, yi - row_h * 0.2), 1, row_h * 0.85, boxstyle="round,pad=0.004",
                    transform=ax_F.transAxes, facecolor="#fff7ed", edgecolor="none", zorder=0))
    gt_mw, ann_mw = row["gt_mw"], row["annotated_mw"]
    delta = ann_mw - gt_mw if (not math.isnan(gt_mw) and not math.isnan(ann_mw)) else float("nan")
    delta_str = f"{delta:+.1f}" if not math.isnan(delta) else "—"
    cells8 = [
        (str(int(row["scan"])), "#555"), (str(row["PRIMARY_NAME"])[:22], "#111"),
        (str(row["top_compound"])[:22] if pd.notna(row["top_compound"]) else "—", "#444"),
        (f"{gt_mw:.1f}" if not math.isnan(gt_mw) else "—", "#333"),
        (f"{ann_mw:.1f}" if not math.isnan(ann_mw) else "—", "#333"),
        (delta_str, "#c05000" if not math.isnan(delta) and abs(delta) > 20 else "#333"),
        (f"{int(row['query_peak_count'])}" if pd.notna(row['query_peak_count']) else "—", "#333"),
        (f"{row['shared_peaks']:.1f}" if pd.notna(row.get('shared_peaks')) else "—", "#333"),
        (f"{row['confidence_label'][:14]}  ({row['confidence_score']:.0f})", CONF_COLOR.get(row["confidence_label"], "#666")),
    ]
    for (v, vc), xp in zip(cells8, col_x):
        ax_F.text(xp, yi, v, transform=ax_F.transAxes, fontsize=7.5, va="top", color=vc)

fig.suptitle(f"Molecular mass × spectral peaks analysis\nTopKonfidence · MSMLS library · task {TASK_ID[:12]}",
             fontsize=13, fontweight="bold", y=1.00)
save_and_show(fig, "mol_mass_report")


## 9. Scoring-parameter correlation matrix

Diagonal: per-confidence-label KDE. Upper triangle: Spearman ρ. Lower triangle: scatter
coloured by confidence label, for the five scoring inputs.

In [ ]:
from scipy import stats
from scipy.stats import gaussian_kde

PARAMS = ["support_fraction" if "adjusted_support_score" not in annex.columns else "adjusted_support_score",
          "tanimoto_score", "cosine_consistency", "max_cosine", "mz_precision"]
LABELS = ["Structure\nagreement", "Tanimoto\nsimilarity", "Cosine\nconsistency", "Max cosine", "M/Z\nprecision"]
WEIGHTS = ["w = 0.50", "w = 0.15", "w = 0.25 (spectral)", "w = 0.25 (spectral)", "multiplier"]
CONF_SHORT9 = ["Consistent", "Inconclusive", "Inconsistent"]
COLORS9 = ["#4daf4a", "#ff7f00", "#e41a1c"]
ALPHAS9 = [0.22, 0.22, 0.22]
POINT_SIZE, SCATTER_N = 4, 600

plt.rcParams.update({"font.size": 8, "axes.linewidth": 0.7, "xtick.major.width": 0.7,
                      "ytick.major.width": 0.7, "axes.spines.top": False, "axes.spines.right": False})

df9 = annex[PARAMS + ["confidence_label"]].dropna()
groups9 = {lab: df9[df9["confidence_label"] == lab] for lab in CONF_ORDER}
N = len(PARAMS)

fig, axes = plt.subplots(N, N, figsize=(7.2, 7.2))
fig.subplots_adjust(left=0.12, right=0.97, top=0.95, bottom=0.12, wspace=0.08, hspace=0.08)
rng9 = np.random.default_rng(42)

for row in range(N):
    for col in range(N):
        ax = axes[row, col]
        xparam, yparam = PARAMS[col], PARAMS[row]
        if row == col:
            for lab, color in zip(CONF_ORDER, COLORS9):
                vals = groups9[lab][xparam].values
                if len(vals) < 2:
                    continue
                x_grid = np.linspace(vals.min() - 0.05, vals.max() + 0.05, 300)
                try:
                    kde = gaussian_kde(vals, bw_method=0.25)
                    ax.plot(x_grid, kde(x_grid), color=color, lw=1.4)
                    ax.fill_between(x_grid, kde(x_grid), alpha=0.18, color=color)
                except Exception:
                    pass
            ax.set_xlim(-0.05, 1.05); ax.set_yticks([]); ax.spines["left"].set_visible(False)
        elif col > row:
            ax.set_axis_off()
            xv, yv = df9[xparam].values, df9[yparam].values
            if len(xv) > 1:
                r, pval = stats.spearmanr(xv, yv)
                norm_r = (r + 1) / 2
                r_color = plt.cm.RdBu_r(norm_r)
                fontsize_r = 9 + 5 * abs(r)
                ax.text(0.5, 0.55, f"ρ = {r:+.2f}", ha="center", va="center", fontsize=fontsize_r,
                        fontweight="bold", color=r_color, transform=ax.transAxes)
                stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
                ax.text(0.5, 0.30, stars, ha="center", va="center", fontsize=7, color="0.45", transform=ax.transAxes)
        else:
            for lab, color, alpha in zip(CONF_ORDER, COLORS9, ALPHAS9):
                grp = groups9[lab]
                n = min(SCATTER_N, len(grp))
                if n == 0:
                    continue
                idx = rng9.choice(len(grp), size=n, replace=False)
                ax.scatter(grp[xparam].values[idx], grp[yparam].values[idx], s=POINT_SIZE, color=color,
                           alpha=alpha, linewidths=0, rasterized=True)
            ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        if row < N - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel(LABELS[col], fontsize=7.5, labelpad=3); ax.tick_params(axis="x", labelsize=6.5, length=3)
        if col > 0:
            ax.set_yticklabels([])
        else:
            ax.set_ylabel(LABELS[row], fontsize=7.5, labelpad=3); ax.tick_params(axis="y", labelsize=6.5, length=3)
        if row != col or col <= row:
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

for i, (ax, wtxt) in enumerate(zip(axes.diagonal(), WEIGHTS)):
    ax.text(0.97, 0.97, wtxt, ha="right", va="top", fontsize=6, color="0.45", transform=ax.transAxes, style="italic")

patches9 = [mpatches.Patch(color=c, label=s, alpha=0.85) for c, s in zip(COLORS9, CONF_SHORT9)]
fig.legend(handles=patches9, loc="upper right", bbox_to_anchor=(0.97, 0.99), frameon=False, fontsize=7.5,
           handlelength=1.2, handleheight=0.9, borderpad=0.4, labelspacing=0.3)

save_and_show(fig, "param_correlation_matrix")


## 10. MS/MS mirror plots

Standalone illustrative pairs of isobaric/isomeric standards (peaks fetched live from the
GNPS2 `metabolomics-usi` endpoint), independent of `TASK_ID`. These are qualitative
"why is this hard" examples, not derived from the scored task.

In [ ]:
import urllib.request, urllib.parse
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from rdkit.Chem import Draw


def fetch_peaks(usi):
    url = "https://metabolomics-usi.gnps2.org/json/?" + urllib.parse.urlencode({"usi1": usi})
    with urllib.request.urlopen(url, timeout=30) as resp:
        d = json.load(resp)
    peaks = np.array(d["peaks"], dtype=float)
    return peaks, d.get("precursor_mz")


def mirror_plot(top_usi, bottom_usi, top_name, bottom_name, top_smiles, bottom_smiles,
                 mz_range, annotate_peaks, out_name, fragment_tol=0.1):
    top_peaks, top_prec = fetch_peaks(top_usi)
    bot_peaks, bot_prec = fetch_peaks(bottom_usi)

    def filter_range(peaks, lo, hi):
        m = (peaks[:, 0] >= lo) & (peaks[:, 0] <= hi)
        return peaks[m]

    top_peaks = filter_range(top_peaks, *mz_range)
    bot_peaks = filter_range(bot_peaks, *mz_range)

    def norm_intensity(peaks):
        return peaks[:, 1] / peaks[:, 1].max() * 65.0

    top_int, bot_int = norm_intensity(top_peaks), norm_intensity(bot_peaks)

    def is_annotated(mz, targets, tol=fragment_tol):
        return any(abs(mz - t) <= tol for t in targets)

    def label_positions(peaks, intensities, targets, tol=fragment_tol):
        out = {}
        for t in targets:
            m = np.abs(peaks[:, 0] - t) <= tol
            if not m.any():
                continue
            idx = np.where(m)[0][np.argmax(intensities[m])]
            out[t] = (peaks[idx, 0], intensities[idx])
        return out

    COLOR_TOP, COLOR_BOTTOM, COLOR_ANNOTATED, STEM_ALPHA = "#1f77b4", "#d62728", "#2ca02c", 0.55

    fig, ax = plt.subplots(figsize=(9.5, 6.8))
    for mz, inten in zip(top_peaks[:, 0], top_int):
        annotated = is_annotated(mz, annotate_peaks)
        color = COLOR_ANNOTATED if annotated else COLOR_TOP
        ax.plot([mz, mz], [0, inten], color=color, linewidth=1.4 if annotated else 1.0,
                alpha=1.0 if annotated else STEM_ALPHA, zorder=3 if annotated else 2)
    for mz, inten in zip(bot_peaks[:, 0], bot_int):
        annotated = is_annotated(mz, annotate_peaks)
        color = COLOR_ANNOTATED if annotated else COLOR_BOTTOM
        ax.plot([mz, mz], [0, -inten], color=color, linewidth=1.4 if annotated else 1.0,
                alpha=1.0 if annotated else STEM_ALPHA, zorder=3 if annotated else 2)

    top_labels, bot_labels = label_positions(top_peaks, top_int, annotate_peaks), label_positions(bot_peaks, bot_int, annotate_peaks)
    for t in annotate_peaks:
        if t in top_labels:
            mz, inten = top_labels[t]
            ax.text(mz, inten + 3, f"{mz:.4f}", ha="center", va="bottom", fontsize=9, color=COLOR_ANNOTATED, rotation=90)
        if t in bot_labels:
            mz, inten = bot_labels[t]
            ax.text(mz, -inten - 3, f"{mz:.4f}", ha="center", va="top", fontsize=9, color=COLOR_ANNOTATED, rotation=90)

    ax.axhline(0, color="#333", linewidth=0.8, zorder=1)
    ax.set_xlim(*mz_range); ax.set_ylim(-82, 82)
    ax.set_xlabel("m/z", fontsize=13); ax.set_ylabel("Relative intensity (%)", fontsize=13)
    ax.set_yticks([-65, -32.5, 0, 32.5, 65]); ax.set_yticklabels(["100", "50", "0", "50", "100"])
    ax.tick_params(axis="both", labelsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.grid(True, linestyle="--", alpha=0.3, zorder=0); ax.set_axisbelow(True)
    fig.subplots_adjust(top=0.82, bottom=0.16, left=0.09, right=0.98)
    fig.text(0.10, 0.955, f"{top_name}   {top_usi.split(':')[-1]}   ·   precursor m/z {top_prec:.4f}",
             ha="left", va="top", fontsize=11, color=COLOR_TOP, fontweight="bold")
    fig.text(0.10, 0.045, f"{bottom_name}   {bottom_usi.split(':')[-1]}   ·   precursor m/z {bot_prec:.4f}",
             ha="left", va="bottom", fontsize=11, color=COLOR_BOTTOM, fontweight="bold")
    handles = [plt.Line2D([0], [0], color=COLOR_TOP, lw=2, label=top_name),
               plt.Line2D([0], [0], color=COLOR_BOTTOM, lw=2, label=bottom_name),
               plt.Line2D([0], [0], color=COLOR_ANNOTATED, lw=2, label="Shared fragment")]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=10, frameon=False)

    def mol_image_array(smiles, size=(220, 180)):
        mol = Chem.MolFromSmiles(smiles)
        return np.array(Draw.MolToImage(mol, size=size))

    ab_top = AnnotationBbox(OffsetImage(mol_image_array(top_smiles), zoom=0.62), (0.985, 0.97),
                             xycoords="axes fraction", box_alignment=(1, 1), frameon=False, pad=0.15)
    ab_bot = AnnotationBbox(OffsetImage(mol_image_array(bottom_smiles), zoom=0.62), (0.985, 0.03),
                             xycoords="axes fraction", box_alignment=(1, 0), frameon=False, pad=0.15)
    ax.add_artist(ab_top); ax.add_artist(ab_bot)

    save_and_show(fig, out_name)


In [ ]:
# Dopamine vs Octopamine
mirror_plot(
    top_usi="mzspec:GNPS:GNPS-LIBRARY:accession:CCMSLIB00005733696",
    bottom_usi="mzspec:GNPS:GNPS-LIBRARY:accession:CCMSLIB00006683690",
    top_name="Dopamine", bottom_name="Octopamine",
    top_smiles="NCCc1ccc(O)c(O)c1", bottom_smiles="NCC(O)c1ccc(O)cc1",
    mz_range=(40.0, 160.0), annotate_peaks=[91.0542, 107.0491, 119.0491],
    out_name="mirror_plot_dopamine_octopamine",
)


In [ ]:
# L-Norvaline vs L-Valine
mirror_plot(
    top_usi="mzspec:GNPS:GNPS-LIBRARY:accession:CCMSLIB00006120370",
    bottom_usi="mzspec:GNPS:GNPS-LIBRARY:accession:CCMSLIB00005885071",
    top_name="L-Norvaline", bottom_name="L-Valine",
    top_smiles="O=C(O)C(N)CCC", bottom_smiles="CC(C)C(C(=O)O)N",
    mz_range=(40.0, 150.0), annotate_peaks=[55.0541, 72.0806, 118.0861],
    out_name="mirror_plot_norvaline_valine",
)
